# Bronze Layer - Data Ingestion

In this step, I load the original CSV file from the Unity Catalog Volume using PySpark. The goal is to keep the raw data before applying any cleaning or transformations.

In [0]:
tickets = spark.read.csv(
    "/Volumes/db_projects/data_engineering/helpdesk_raw/synthetic_it_support_tickets.csv",
    header=True,
    inferSchema=True
)

display(tickets.head(5))

ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
TCKT_000001,2024-01-31T05:14:27.000Z,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,0,android,EU
TCKT_000002,2024-10-20T06:15:49.000Z,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,0,web,NA
TCKT_000003,2024-06-18T21:35:54.000Z,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my changes.,Thanks for reporting this bug. We will look into this and get back to you within the next 48 hours.,null,null,0,neutral,3,0,android,MEA
TCKT_000004,2025-12-25T15:59:52.000Z,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. Our team is reviewing this and will follow up within 24 hours.,null,null,0,positive,5,1,android,LATAM
TCKT_000005,2023-08-27T16:08:33.000Z,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you within the next 48 hours.,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,0,web,NA


## Create Bronze Table

I add an ingestion timestamp to track when the data was loaded, then save the dataset as a Delta table in Databricks. This table will be used as the source for the Silver layer.

In [0]:
from pyspark.sql import functions as F

(tickets.withColumn("ingestion_timestamp", F.current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable("db_projects.data_engineering.ID001_Bronze_Tickets"))

## Verify that the table was created and has the data inserted before

In [0]:
bronze = spark.table("db_projects.data_engineering.ID001_Bronze_Tickets")

print("Total records:", bronze.count())
display(bronze.limit(5))

Total records: 100000


ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region,ingestion_timestamp
TCKT_000001,2024-01-31T05:14:27.000Z,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours.,Reset account credentials and confirmed successful login.,36.53,0,very_negative,1,0,android,EU,2026-10-28T11:25:54.474Z
TCKT_000002,2024-10-20T06:15:49.000Z,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is reviewing this and will follow up within 24 hours.,Ticket closed without further action after no response from the customer.,238.32,0,neutral,3,0,web,NA,2026-10-28T11:25:54.474Z
TCKT_000003,2024-06-18T21:35:54.000Z,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my changes.,Thanks for reporting this bug. We will look into this and get back to you within the next 48 hours.,null,null,0,neutral,3,0,android,MEA,2026-10-28T11:25:54.474Z
TCKT_000004,2025-12-25T15:59:52.000Z,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password is incorrect.,Sorry to hear you're having trouble accessing your account. Our team is reviewing this and will follow up within 24 hours.,null,null,0,positive,5,1,android,LATAM,2026-10-28T11:25:54.474Z
TCKT_000005,2023-08-27T16:08:33.000Z,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the plan I selected.,Thanks for reaching out about the billing issue. We will look into this and get back to you within the next 48 hours.,Adjusted the invoice and issued a refund where applicable.,61.32,0,very_negative,2,0,web,NA,2026-10-28T11:25:54.474Z
